# MobileADAS3D — MobileNetV4 Conv Small baseline

This notebook trains a fresh monocular-3D baseline on the canonical KITTI Chen 3,712/3,769 split. It keeps the existing stride-16 FPN and eight 3D heads, changes only the backbone to pretrained MobileNetV4 Conv Small, saves checkpoints to Google Drive, resumes after disconnects, and produces KITTI BEV/3D AP_R40 artifacts.

Before running: select **Runtime → Change runtime type → GPU** and make sure the repository changes containing this notebook are pushed to GitHub.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
import json, os, shlex, shutil, subprocess, sys, time

REPO_URL = 'https://github.com/Ali-RT/mobile_adas3d.git'
BRANCH = 'main'
PROJECT_DIR = Path('/content/mobile_adas3d')
CONFIG = 'configs/kitti_mnv4_conv_small_baseline.yaml'
DRIVE_DATASET_ROOT = Path('/content/drive/MyDrive/datasets/kitti')
RUNTIME_DATASET_ROOT = Path('/content/kitti')
SPLIT_DIR = Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
OUTPUT_DIR = Path('/content/drive/MyDrive/mobile_adas3d_outputs/mnv4_conv_small_baseline')
STAGE_DATA_TO_LOCAL = True  # Faster epochs; source data remains in Drive.
FORCE_RESTAGE_DATA = False  # Set True only when you want rsync to repair/re-copy local KITTI.
AUTO_RESUME = True

def run(command, cwd=None):
    print('+', ' '.join(shlex.quote(str(x)) for x in command))
    completed = subprocess.run([str(x) for x in command], cwd=cwd)
    if completed.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {completed.returncode}: {command}')


In [ ]:
if not (PROJECT_DIR / '.git').exists():
    run(['git', 'clone', '--branch', BRANCH, REPO_URL, PROJECT_DIR])
else:
    run(['git', 'fetch', 'origin'], cwd=PROJECT_DIR)
    run(['git', 'checkout', BRANCH], cwd=PROJECT_DIR)
    run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=PROJECT_DIR)
os.chdir(PROJECT_DIR)
print('Repository:', PROJECT_DIR)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], cwd=PROJECT_DIR)
import torch, timm
print('torch:', torch.__version__)
print('timm:', timm.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled. Select Runtime > Change runtime type > GPU.')
print('GPU:', torch.cuda.get_device_name(0))


## Stage KITTI onto the Colab runtime

Training directly from mounted Drive can bottleneck the data loader. This cell stages KITTI to temporary Colab storage for faster epochs while checkpoints and metrics remain in Drive. It prints Drive path diagnostics, source/local counts, local disk space, a per-folder file-count progress bar, and writes `/content/kitti/.mobileadas3d_stage_manifest.json`. If Colab disconnects during staging, rerun this cell. Set `STAGE_DATA_TO_LOCAL=False` only if local disk space is insufficient.

In [ ]:
KITTI_REQUIRED_SUBDIRS = {
    'training/image_2': '.png',
    'training/label_2': '.txt',
    'training/calib': '.txt',
}
KITTI_EXPECTED_COUNT = 7481
STAGE_MANIFEST = RUNTIME_DATASET_ROOT / '.mobileadas3d_stage_manifest.json'

def count_files(directory, suffix):
    if not directory.is_dir():
        return 0
    return sum(1 for path in directory.iterdir() if path.is_file() and path.suffix == suffix)

def collect_counts(root):
    return {subdir: count_files(root / subdir, suffix) for subdir, suffix in KITTI_REQUIRED_SUBDIRS.items()}

def print_counts(title, root, counts):
    print(f'\n{title}: {root}')
    for subdir in KITTI_REQUIRED_SUBDIRS:
        print(f'  {subdir}: {counts[subdir]}')

def list_directory(path, limit=30):
    if not path.exists():
        print(f'  {path} does not exist')
        return
    if not path.is_dir():
        print(f'  {path} exists but is not a directory')
        return
    names = sorted(child.name for child in path.iterdir())[:limit]
    print(f'  {path}: {names}')

def validate_kitti_root(root, label):
    missing = [subdir for subdir in KITTI_REQUIRED_SUBDIRS if not (root / subdir).is_dir()]
    if missing:
        print(f'\n{label} is missing required KITTI folders: {missing}')
        print('Nearby directories to inspect:')
        list_directory(root)
        list_directory(root / 'training')
        raise FileNotFoundError(f'{label} missing KITTI folders under {root}')
    counts = collect_counts(root)
    print_counts(f'{label} counts', root, counts)
    bad = {subdir: count for subdir, count in counts.items() if count != KITTI_EXPECTED_COUNT}
    if bad:
        raise RuntimeError(f'{label} expected {KITTI_EXPECTED_COUNT} files in each folder, got {bad}')
    return counts

def local_disk_free_gb(path):
    usage_path = path if path.exists() else path.parent
    usage = shutil.disk_usage(usage_path)
    return usage.free / (1024 ** 3), usage.total / (1024 ** 3)

def write_stage_manifest(source_counts, staged_counts, complete):
    RUNTIME_DATASET_ROOT.mkdir(parents=True, exist_ok=True)
    payload = {
        'complete': complete,
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'source': str(DRIVE_DATASET_ROOT),
        'destination': str(RUNTIME_DATASET_ROOT),
        'expected_count_per_folder': KITTI_EXPECTED_COUNT,
        'required_subdirs': KITTI_REQUIRED_SUBDIRS,
        'source_counts': source_counts,
        'staged_counts': staged_counts,
    }
    STAGE_MANIFEST.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n')
    print('Stage manifest:', STAGE_MANIFEST)

def stage_is_complete():
    if not STAGE_MANIFEST.is_file():
        return False
    try:
        manifest = json.loads(STAGE_MANIFEST.read_text())
    except json.JSONDecodeError:
        return False
    if manifest.get('complete') is not True:
        return False
    return all(count == KITTI_EXPECTED_COUNT for count in collect_counts(RUNTIME_DATASET_ROOT).values())

def copy_subdir_with_progress(subdir, suffix):
    from tqdm.auto import tqdm

    source_dir = DRIVE_DATASET_ROOT / subdir
    destination_dir = RUNTIME_DATASET_ROOT / subdir
    destination_dir.mkdir(parents=True, exist_ok=True)
    initial = min(count_files(destination_dir, suffix), KITTI_EXPECTED_COUNT)
    command = [
        'rsync',
        '-ah',
        '--partial',
        '--stats',
        f'{source_dir}/',
        f'{destination_dir}/',
    ]
    print('\nCopying:', subdir)
    print('+', ' '.join(shlex.quote(str(x)) for x in command))
    process = subprocess.Popen(command)
    last = initial
    with tqdm(total=KITTI_EXPECTED_COUNT, initial=initial, desc=subdir, unit='files', dynamic_ncols=True) as progress:
        while process.poll() is None:
            time.sleep(1.0)
            current = min(count_files(destination_dir, suffix), KITTI_EXPECTED_COUNT)
            progress.update(max(0, current - last))
            last = current
        current = min(count_files(destination_dir, suffix), KITTI_EXPECTED_COUNT)
        progress.update(max(0, current - last))
    if process.returncode != 0:
        raise RuntimeError(f'rsync failed for {subdir} with exit code {process.returncode}. Rerun this cell to resume, or inspect the rsync output above.')
    print(f'Finished {subdir}: {count_files(destination_dir, suffix)}/{KITTI_EXPECTED_COUNT} files')

print('Drive dataset root:', DRIVE_DATASET_ROOT)
print('Runtime dataset root:', RUNTIME_DATASET_ROOT)
print('rsync path:', shutil.which('rsync'))
free_gb, total_gb = local_disk_free_gb(RUNTIME_DATASET_ROOT)
print(f'Local disk free: {free_gb:.1f} GB / {total_gb:.1f} GB')
print('\nDrive directory check:')
list_directory(DRIVE_DATASET_ROOT.parent)
list_directory(DRIVE_DATASET_ROOT)

if STAGE_DATA_TO_LOCAL:
    if shutil.which('rsync') is None:
        raise RuntimeError('rsync is not available in this Colab runtime.')
    source_counts = validate_kitti_root(DRIVE_DATASET_ROOT, 'Drive KITTI source')
    staged_counts = collect_counts(RUNTIME_DATASET_ROOT)
    print_counts('Current local staged counts', RUNTIME_DATASET_ROOT, staged_counts)
    if not FORCE_RESTAGE_DATA and stage_is_complete():
        print('\nLocal /content/kitti stage is already complete. Skipping copy.')
    else:
        write_stage_manifest(source_counts, staged_counts, complete=False)
        for subdir, suffix in KITTI_REQUIRED_SUBDIRS.items():
            copy_subdir_with_progress(subdir, suffix)
        staged_counts = validate_kitti_root(RUNTIME_DATASET_ROOT, 'Local staged KITTI')
        write_stage_manifest(source_counts, staged_counts, complete=True)
    DATASET_ROOT = RUNTIME_DATASET_ROOT
else:
    source_counts = validate_kitti_root(DRIVE_DATASET_ROOT, 'Drive KITTI source')
    DATASET_ROOT = DRIVE_DATASET_ROOT
print('\nTraining dataset root:', DATASET_ROOT)


## Canonical split and full preflight

This fails before training unless all 7,481 images, labels, and calibration files exist; the split is exactly 3,712/3,769 with no overlap; CUDA works; pretrained MobileNetV4 loads; output shapes remain stride 16; and a real KITTI loss is finite.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
run([sys.executable, 'scripts/prepare_kitti_chen_split.py', '--config', CONFIG, '--profile', 'colab_drive', '--output-dir', SPLIT_DIR], cwd=PROJECT_DIR)
COMMON = ['--config', CONFIG, '--profile', 'colab_drive', '--dataset-root', DATASET_ROOT, '--split-dir', SPLIT_DIR, '--output-dir', OUTPUT_DIR]
run([sys.executable, 'scripts/check_kitti_splits.py', *COMMON], cwd=PROJECT_DIR)
run([sys.executable, 'scripts/check_training_ready.py', *COMMON, '--require-cuda', '--report', OUTPUT_DIR / 'training_preflight.json'], cwd=PROJECT_DIR)


## Train or resume

`latest.pt` is atomically replaced after every epoch. With `AUTO_RESUME=True`, rerunning this cell after a Colab disconnect continues the newest run in this dedicated output directory. Epoch snapshots are retained every 10 epochs.

In [ ]:
latest_candidates = list((OUTPUT_DIR / 'runs').glob('*/checkpoints/latest.pt'))
resume_checkpoint = max(latest_candidates, key=lambda p: p.stat().st_mtime) if (AUTO_RESUME and latest_candidates) else None
train_command = [sys.executable, 'scripts/train_mobile_adas3d.py', *COMMON]
if resume_checkpoint is not None:
    train_command += ['--resume', resume_checkpoint]
    TRAIN_RUN_DIR = resume_checkpoint.parent.parent
    print('Resuming:', resume_checkpoint)
else:
    print('Starting a new baseline run')
run(train_command, cwd=PROJECT_DIR)
if resume_checkpoint is None:
    run_dirs = list((OUTPUT_DIR / 'runs').glob('*'))
    TRAIN_RUN_DIR = max(run_dirs, key=lambda p: p.stat().st_mtime)
BEST_CHECKPOINT = TRAIN_RUN_DIR / 'checkpoints' / 'best.pt'
if not BEST_CHECKPOINT.is_file():
    raise FileNotFoundError(f'Best checkpoint missing: {BEST_CHECKPOINT}')
print('Run directory:', TRAIN_RUN_DIR)
print('Best checkpoint:', BEST_CHECKPOINT)


## Generate the reportable KITTI baseline

This evaluates all 3,769 validation images at a low score floor and writes `kitti_r40_metrics.csv`, `kitti_r40_summary.json`, and raw KITTI-format predictions. Only results with `complete_split: true` are reportable.

In [ ]:
EVAL_DIR = TRAIN_RUN_DIR / 'kitti_r40_val'
run([sys.executable, 'scripts/evaluate_kitti_r40.py', '--config', CONFIG, '--profile', 'colab_drive', '--dataset-root', DATASET_ROOT, '--split-dir', SPLIT_DIR, '--checkpoint', BEST_CHECKPOINT, '--split', 'val', '--score-threshold', '0.001', '--topk', '300', '--nms-iou-threshold', '0.5', '--output-dir', EVAL_DIR], cwd=PROJECT_DIR)

import json, pandas as pd
summary = json.loads((EVAL_DIR / 'kitti_r40_summary.json').read_text())
assert summary['complete_split'] and summary['evaluated_images'] == 3769
metrics = pd.DataFrame(summary['metrics'])
display(metrics.pivot_table(index=['metric', 'class_name'], columns='difficulty', values='ap_r40').round(3))
print('Baseline artifacts:', EVAL_DIR)


## Optional TensorBoard

Run the following cell while training or after it finishes.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/mobile_adas3d_outputs/mnv4_conv_small_baseline/runs
